# Transaction Foundation Model on Ray — Part 2: Load & explore the data w/ Ray Data

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~15 min (most of it the one-time dataset download + normalize)


---

In Part 1, we set up the cluster. Here in Part 2, we load the benchmark dataset

In [1]:
import sys, os, json

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import logging
import seaborn as sns

import ray
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT},   logging_level=logging.ERROR)

/home/ray/anaconda3/lib/python3.12/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Python version:,3.12.13
Ray version:,2.55.1
Dashboard:,http://session-nridlis6nyy2hi9qv4il5lcj6q.i.anyscaleuserdata.com


## IBM TabFormer Dataset

We use the **IBM TabFormer** dataset (Padhi et al., ICASSP 2021) — the de-facto public benchmark for transaction foundation models. It has 24.4M card transactions, ~6.1k cards, 1991–2020, ~0.12% transaction-level fraud. **NVIDIA's transaction-FM blueprint** and FATA-Trans both evaluate on it, and this series reproduces NVIDIA's result on Ray — so our numbers are directly comparable to theirs.

We build the train/val/test split the **same way NVIDIA does**, straight from the raw CSV (no precomputed files):

1. **Download** the raw `card_transaction.v1.csv` once (~2.3 GB), cached under `source/`.
2. **Temporal 80/10/10 split** by *cumulative transaction date* — the model trains on the earliest 80% of transactions, validates on the next 10%, and is tested on the most recent 10%. Splitting by time (not randomly) means the model is always evaluated on the *future*, so it can't peek ahead.
3. **100K stratified eval subsets** of val and test that preserve the natural ~0.1% fraud rate — NVIDIA's `val_eval` / `test_eval`. Scoring 100K rows instead of millions keeps evaluation cheap without distorting the metric.

This runs as a single cuDF/GPU job in `src/nvsplit.py` (mirroring NVIDIA's NB01), writing TabFormer's **native columns** — exactly what the tokenizer in Part 3 consumes.

### A note on `SCALE` config

`SCALE` is the one knob that defines a run. Here it controls how many card-holders to include and the eval-subset size; in later notebooks, the sequence length, model size, and number of GPU workers. The presets (`mini` / `small` / `full`) live in `configs/`. **`mini`** keeps only a couple hundred card-holders for a fast sanity run; **`full`** uses every card. It's the *same code, change config* idea — flip the variable to go from a smoke test to the full multi-GPU run. Outputs are written under a per-scale path (`.../nvsplit/<scale>/`) so scales don't overwrite each other.

### Build the split from the raw CSV

The next cell downloads the CSV (once) and regenerates NVIDIA's temporal split + stratified eval subsets in `src/nvsplit.py`. Re-runs reuse the cached parquet.

In [ ]:
from src.paths import artifact_paths, get_demo_base_dir
from src.scale_config import load_scale
from src.tabformer import ensure_download
from src.nvsplit import build_temporal_split

SCALE = "full"            # which configs/<SCALE>.yaml preset to run; mini = tiny & fast
cfg = load_scale(SCALE)    # configs/<SCALE>.yaml — data knobs here; model/training in later parts
paths = artifact_paths(get_demo_base_dir(), SCALE)   # outputs namespaced per scale: .../nvsplit/<scale>/

# Regenerate the temporal split straight from the raw TabFormer CSV — NO precomputed artifacts.
# This mirrors NVIDIA's data-prep (their NB01) exactly: an 80/10/10 split by *cumulative
# transaction date* (so val/test are the most-recent transactions → no leakage), plus 100K
# stratified val/test eval subsets that preserve the natural ~0.1% fraud rate. Output keeps
# TabFormer's native columns — exactly what the tokenizer in Part 3 consumes.
# (src/nvsplit.py runs this as a single cuDF/GPU task; re-runs reuse the cached parquet.)
if not os.path.exists(os.path.join(paths["nvsplit"], "train.parquet")):
    csv_path = ensure_download(paths["source"])          # one-time ~2.3GB CSV download + extract
    meta = build_temporal_split(
        csv_path, paths["nvsplit"],
        eval_samples=cfg["data"]["eval_samples"],        # 100K stratified val/test
        max_users=cfg["data"]["max_users"],              # None = every card-holder (full)
    )
    print(json.dumps(meta, indent=2))
else:
    print("nvsplit already built at", paths["nvsplit"])

In [ ]:
# Load the temporal-TRAIN split (native TabFormer columns) and derive a few convenience
# columns for exploration. The tokenizer in Part 3 uses the native columns directly.
df = pd.read_parquet(os.path.join(paths["nvsplit"], "train.parquet"))
df["card_id"]     = df["User"] * 1000 + df["Card"]          # one card-holder+card = one sequence
df["is_fraud"]    = (df["Is Fraud?"].astype(str).str.lower() == "yes").astype(int)
df["amount"]      = (df["Amount"].astype(str).str.replace("$", "", regex=False)
                                  .str.replace(",", "", regex=False).astype(float))
df["hour"]        = df["Time"].astype(str).str.split(":", n=1).str[0].astype(int)
df["timestamp"]   = pd.to_datetime(
    df["Year"].astype(str) + "-" + df["Month"].astype(str).str.zfill(2) + "-"
    + df["Day"].astype(str).str.zfill(2) + " " + df["Time"].astype(str), errors="coerce")
df["day_of_week"] = df["timestamp"].dt.dayofweek

with open(os.path.join(paths["nvsplit"], "split_meta.json")) as f:
    split_meta = json.load(f)

print(f"train split: {len(df):,} transactions  /  {df['card_id'].nunique():,} cards  "
      f"/  {df['is_fraud'].mean()*100:.3f}% fraud")
df[["User", "Card", "timestamp", "amount", "Merchant Name", "MCC",
    "Use Chip", "Merchant State", "is_fraud"]].head(5)

## One card is one sequence

The model learns a card's transactions as a time-ordered sequence. Here is what one looks like, these are the dynamic fields the model will learn to predict (masked-feature modeling, Part 4).

In [ ]:
card = df["card_id"].iloc[0]
seq = df[df["card_id"] == card].sort_values("timestamp")
print(f"card {card}: {len(seq):,} transactions from "
      f"{seq['timestamp'].min().date()} to {seq['timestamp'].max().date()}")
seq[["timestamp", "amount", "Merchant Name", "MCC", "Merchant State",
     "hour", "day_of_week", "is_fraud"]].head(10)

## Dataset shape and the temporal split

A final review of the data stats and temporal splits. 

In [ ]:
seq_lens = df.groupby("card_id").size()
print(f"train transactions ...... {len(df):,}")
print(f"cards ................... {df['card_id'].nunique():,}")
print(f"txns per card ........... median {int(seq_lens.median()):,}  "
      f"(min {seq_lens.min():,}, max {seq_lens.max():,})")
print(f"train date range ........ {df['timestamp'].min().date()} -> {df['timestamp'].max().date()}")
print(f"train fraud rate ........ {df['is_fraud'].mean()*100:.3f}%  "
      f"({int(df['is_fraud'].sum()):,} fraudulent txns)")
print()
print("temporal split cutoffs (src/nvsplit.py -> split_meta.json — the same split every stage uses):")
print(f"  train  <  {split_meta['train_cutoff']}")
print(f"  test   >= {split_meta['test_cutoff']}")
print(f"  eval subsets: val {split_meta['val_eval']['rows']:,} rows, "
      f"test {split_meta['test_eval']['rows']:,} rows "
      f"(stratified, ~{split_meta['test_eval']['fraud_rate']*100:.2f}% fraud)")

## Inspecting Data Distributions

Before we build out a tokenizer or a model, we inspect the data with a few plots. Each one guides an upcoming design decision.

1. **Transactions per card.** Cards vary enormously in activity — from just a handful of transactions to nearly 50,000, with a typical (median) card around 2,500. That huge spread is what the *sequence length* has to handle: a window long enough to capture real behavioral context for active cards, but not so long that the many short-history cards (roughly a fifth have fewer than 100 transactions) become mostly padding.
2. **Transaction amount.** Amounts span several orders of magnitude — from under a dollar to a few thousand — but cluster in the tens of dollars (median ~\$33), with a right-skewed tail of larger purchases. Because amount is multiplicative rather than additive (the step from \$10 to \$100 matters like \$100 to \$1,000), a raw scalar would spend almost all its resolution on the crowded low end. So we *bucket amounts on a log scale* and treat each bucket as its own categorical token.
3. **Volume over time.** Transaction volume grows steeply through the 2000s and then levels off, so an 80/10/10 split by transaction count puts the validation and test cutoffs in the most recent stretch of calendar time. That's what we want: splitting by time trains the model on the past and evaluates it on the future, so it never gets to peek ahead.
4. **Time between transactions.** The gaps are bursty — often only a few hours apart (median ~9 hours, a fifth of them within the hour), but stretching to days or, occasionally, far longer. So a position's ordinal slot ("the 5th transaction") throws away how much real time actually passed; we also embed *how long it's been since the previous transaction*, on a log scale, to make position time-aware.

In [ ]:
sns.set_theme(style="white", context="notebook")
from matplotlib.ticker import FuncFormatter

# Human-readable axis numbers: 600000 -> "600k", 2_000_000 -> "2M".
def _human(x, _):
    x = float(x)
    if abs(x) >= 1_000_000: return f"{x / 1e6:g}M"
    if abs(x) >= 1_000:     return f"{x / 1e3:g}k"
    return f"{x:g}"
human = FuncFormatter(_human)

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# (1) transactions per card -> motivates the sequence window length (seq_len).
ax = axes[0, 0]
sns.histplot(seq_lens, bins=40, color="#4C78A8", ax=ax)
ax.set_yscale("log")
ax.set_title("Transactions per card (sequence length)")
ax.set_xlabel("transactions"); ax.set_ylabel("cards (log scale)")
ax.xaxis.set_major_formatter(human); ax.yaxis.set_major_formatter(human)

# (2) amount spans orders of magnitude -> the tokenizer bins it (optimal binning).
ax = axes[0, 1]
log_amt = np.log10(np.clip(np.abs(df["amount"].to_numpy()), 0.1, None))
sns.histplot(log_amt, bins=60, color="#F58518", ax=ax)
ax.set_title("Transaction amount (log10 $) — spans orders of magnitude")
ax.set_xlabel("log10(amount)"); ax.set_ylabel("transactions")
ax.yaxis.set_major_formatter(human)

# (3) train volume over time, with the train|val temporal cutoff (test = later, not shown here).
ax = axes[1, 0]
monthly = df.groupby(df["timestamp"].dt.to_period("M")).size()
sns.lineplot(x=monthly.index.to_timestamp(), y=monthly.values, color="#54A24B", ax=ax)
cut = pd.Timestamp(split_meta["train_cutoff"])
ax.axvline(cut, color="crimson", ls="--", lw=1)
ax.text(cut, ax.get_ylim()[1] * 0.92, "train|val", rotation=90,
        va="top", ha="right", color="crimson", fontsize=8)
ax.set_title("Train transaction volume over time (+ train|val cutoff)")
ax.set_xlabel("month"); ax.set_ylabel("transactions")
ax.yaxis.set_major_formatter(human)

# (4) inter-transaction gaps -> motivates the tokenizer's temporal encoding.
ax = axes[1, 1]
gaps = df.sort_values(["card_id", "timestamp"]).groupby("card_id")["timestamp"].diff().dt.total_seconds() / 3600.0
gaps = gaps.dropna()
gaps = gaps[gaps > 0]
sns.histplot(np.log10(gaps), bins=60, color="#B279A2", ax=ax)
ax.set_title("Hours between consecutive txns (log10) — bursty")
ax.set_xlabel("log10(hours since previous txn)"); ax.set_ylabel("transactions")
ax.yaxis.set_major_formatter(human)

sns.despine(fig=fig)
plt.tight_layout()
plt.show()

## Evaluations & how we measure performance

With only about 1 transaction in 800 fraudulent, 'accuracy' is meaningless — a model that flags everything as valid will score 99.9% accuracy and catches no fraud. We use PR-AUC (average precision) instead, which rewards ranking the rare fraud above everything else.

That same rarity shapes the eval set: we keep every fraud but drop most normal transactions, since scoring millions of obvious non-fraud just wastes compute. That leaves the set far more fraud-heavy than reality, so we apply 'importance weighting', which normalizes the evaluation metrics at the true ~0.12% rate.

In [8]:
fraud_rate = df["is_fraud"].mean()
print(f"normal transactions : {(1 - fraud_rate) * 100:.3f}%")
print(f"fraud transactions  : {fraud_rate * 100:.3f}%")
print(f"imbalance ratio     : ~1 fraud per {int(round(1 / fraud_rate)):,} transactions")

normal transactions : 99.878%
fraud transactions  : 0.122%
imbalance ratio     : ~1 fraud per 820 transactions


## Takeaways

**Ray**
- The split is regenerated from the raw CSV as a single cuDF/GPU job (`src/nvsplit.py`), mirroring NVIDIA's NB01 — no dependency on precomputed artifacts.
- The same code runs at any scale: `mini` (a couple hundred card-holders, fast) → `full` (every card); only the config changes.

**The data**
- One card = one **time-ordered sequence** — the unit the foundation model learns over (Parts 3–4).
- **Temporal 80/10/10** split by cumulative transaction date (test = most recent), plus **100K stratified** val/test eval subsets at the natural fraud rate — identical to NVIDIA's protocol, so our numbers are comparable to theirs.
- The distributions drive the tokenizer choices in Part 3: amounts span orders of magnitude (optimal binning), inter-transaction gaps are bursty (temporal encoding), and card histories vary enormously (windowed sequences).
- Fraud is rare (~1 in 800), so we measure with PR-AUC (average precision), not accuracy.

---

## Next

**Part 3 — Tokenize**: turn each card's native transaction rows into token sequences with NVIDIA's `FinancialTabularTokenizer` (merchant hashing + category hierarchy + temporal encoding, vocab 6251) and build the 4096-token pretraining corpus.